# facetrust — demo hands-on (Colab)

Tutorial: *Biometrics under attack: On provenance, trust and threat robustness*

**Duração alvo: 20–30 minutos**, dentro do bloco de 90 min pós-café ("Joint evaluation").

**Antes da sessão** (não ao vivo, para não gastar o tempo do grupo com downloads):
1. Rode a célula de **setup** (próxima seção) para confirmar que o ambiente sobe sem erro.
2. Os arquivos `facetrust.zip` (o pacote), `data.zip` (as imagens limpas e marcadas) e `morphs.zip` (as morphs do FRLL) são baixados do Google Drive pela própria célula de setup -- não precisa subir nada manualmente. Se algum download falhar (cota do Drive), a célula pede o upload manual como plano B.
3. As morphs vêm num zip separado ([morphs.zip](https://drive.google.com/file/d/168zRvzdFNWKUS0hVgfam4RjWOUfR1Gr3/view?usp=sharing)). A célula de setup extrai esse arquivo em `morphs_src/` e roda `scripts/copy_frll_data.py --skip-clean --morph-src <pasta> --datadir data`, que copia as imagens para `data/morph/<algoritmo>/<a>+<b>.png` (a convenção que o loader espera). Rodando localmente, faça o mesmo: baixe e extraia o `morphs.zip` e aponte `--morph-src` para a pasta que contém as subpastas `morph_*`.

**Sobre o dataset:** recomendamos o *Face Research Lab London Set* (FRLL) para as imagens "clean" --
licença CC BY 4.0, consentimento explícito de uso em pesquisa (https://figshare.com/articles/dataset/Face_Research_Lab_London_Set/5047666). Para as morphs, o **FRLL-Morphs** já traz pares pré-gerados por vários algoritmos (AMSL, FaceMorpher, OpenCV, StyleGAN2 -- https://www.idiap.ch/en/scientific-research/data/frll-morphs), o que é ótimo para a Seção 6 (comparar algoritmos de morphing lado a lado).

**Ambiente recomendado:** Runtime > Change runtime type > GPU (acelera o ArcFace via `insightface`; CPU também funciona para um dataset pequeno).

## 0. Setup (rodar antes da sessão, se possível)

In [ ]:
# Baixa facetrust.zip, data.zip e morphs.zip do Google Drive (via gdown, já vem no Colab).
# Extraímos o pacote para 'src/' de propósito: extrair facetrust.zip direto no
# diretório de trabalho criaria uma pasta ./facetrust/ que colide com o pacote
# python 'facetrust' (o Python trata ./facetrust como um "namespace package"
# vazio antes de achar o pacote de verdade lá dentro) -- daria erros estranhos
# tipo 'module facetrust has no attribute...' em QUALQUER célula depois desta.
import os, zipfile

FACETRUST_ID = '1uVWhPfZTaQkec84Pn32ZdioAAaOFTOnS'
DATA_ID = '17cANlhPb-3H9D6EGRadIOEYfXIKBKZwk'
MORPHS_ID = '168zRvzdFNWKUS0hVgfam4RjWOUfR1Gr3'

def get_zip(file_id, filename):
    """Baixa do Drive; se falhar (ex. cota de downloads do Drive estourada
    numa sala cheia), cai para upload manual do arquivo."""
    try:
        import gdown
        gdown.download(url=f'https://drive.google.com/uc?id={file_id}', output=filename, quiet=False)
        if not os.path.exists(filename):
            raise RuntimeError('download não gerou o arquivo')
    except Exception as exc:
        print(f'Falha ao baixar {filename} do Drive ({exc}).')
        print(f'Plano B: selecione o arquivo {filename} do seu computador.')
        from google.colab import files
        uploaded = files.upload()
        name = filename if filename in uploaded else next(iter(uploaded))
        if name != filename:
            os.replace(name, filename)

get_zip(FACETRUST_ID, 'facetrust.zip')
get_zip(DATA_ID, 'data.zip')
get_zip(MORPHS_ID, 'morphs.zip')

os.makedirs('src', exist_ok=True)
with zipfile.ZipFile('facetrust.zip') as zf:
    zf.extractall('src')

# data.zip: se o zip já traz a pasta 'data/' na raiz, extrai em '.'; senão, em 'data/'.
with zipfile.ZipFile('data.zip') as zf:
    top_level_is_data = all(n == 'data' or n.startswith('data/') for n in zf.namelist())
    zf.extractall('.' if top_level_is_data else 'data')

# morphs.zip: extrai em 'morphs_src/' e usa scripts/copy_frll_data.py para
# copiar as morphs para data/morph/<algoritmo>/<a>+<b>.png. O script espera
# uma pasta com subpastas 'morph_<algoritmo>' -- procuramos essa pasta
# automaticamente, esteja ela na raiz do zip ou um nível abaixo.
import glob, subprocess, sys
with zipfile.ZipFile('morphs.zip') as zf:
    zf.extractall('morphs_src')

candidates = sorted({os.path.dirname(d.rstrip('/')) for d in
                     glob.glob('morphs_src/morph_*/') + glob.glob('morphs_src/*/morph_*/')})
copy_script = 'src/facetrust/scripts/copy_frll_data.py'
if not candidates:
    print('Nenhuma subpasta morph_* encontrada em morphs_src/ -- confira o conteúdo de morphs.zip.')
elif not os.path.exists(copy_script):
    print(f'{copy_script} não encontrado -- o facetrust.zip baixado é de uma versão antiga.')
else:
    morph_src = candidates[0]
    morph_types = sorted(os.path.basename(d.rstrip('/')) for d in glob.glob(f'{morph_src}/morph_*/'))
    print('Morphs em', morph_src, '->', morph_types)
    subprocess.run([sys.executable, copy_script, '--skip-clean',
                    '--morph-src', morph_src, '--datadir', 'data',
                    '-t', *morph_types], check=True)

# Compatibilidade: algumas versões do demo_full_pipeline.py leem data/morph/,
# outras data/morphed/. Deixa os dois nomes apontando para as mesmas imagens.
if os.path.isdir('data/morph') and not os.path.exists('data/morphed'):
    os.symlink('morph', 'data/morphed')
elif os.path.isdir('data/morphed') and not os.path.exists('data/morph'):
    os.symlink('morphed', 'data/morph')

print('src/ :', os.listdir('src'))
print('data/:', os.listdir('data'))

In [ ]:
# Instala o facetrust em modo editável (usa o pyproject.toml do pacote --
# traz numpy/scipy/scikit-learn/scikit-image/pandas/matplotlib/Pillow como
# dependências core, mais o extra [arcface] para o ArcFaceEmbedder pronto).
# Em Colab isso costuma levar 1-3 minutos na primeira vez (baixa os pesos
# do insightface na primeira chamada de ArcFaceEmbedder(), não aqui).
%pip install -q -e "src/facetrust[arcface]"

import sys
sys.path.insert(0, 'src/facetrust/examples')  # para reaproveitar load_dataset() do demo_full_pipeline.py

In [ ]:
# Teste rápido de que o pacote foi importado corretamente (sem depender de
# modelos ou imagens reais) -- bom "aquecimento" enquanto o resto do grupo
# termina o upload.
import numpy as np
import facetrust
from facetrust.verification import cosine_similarity
from facetrust.image_quality import psnr, ssim
from facetrust.morphing import mmpmr, fmmpmr, prodavg_mmpmr

print('facetrust', facetrust.__version__)
v = np.random.rand(512)
print('cosine(v, v) =', cosine_similarity(v, v))
img = (np.random.rand(64, 64, 3) * 255).astype('uint8')
print('psnr(img, img) =', psnr(img, img), ' ssim(img, img) =', ssim(img, img))
# MMPMR/FMMPMR com múltiplas provas por contribuinte -- devem DIVERGIR
# (essa é a distinção entre as duas; se baterem igual com só 1 prova, tudo bem)
toy = {'m1': {'a': [0.95, 0.40], 'b': [0.90, 0.88]}}
print('mmpmr:', mmpmr(toy, 0.86), ' fmmpmr:', fmmpmr(toy, 0.86), ' prodavg:', prodavg_mmpmr(toy, 0.86))
print('\nSetup OK.')

## 1. Extrator de embeddings

Tenta usar ArcFace via `insightface` (baixa os pesos na primeira chamada -- 
por isso a etapa de setup deve rodar **antes** da sessão). Se não houver 
internet/GPU disponível no momento, cai para um embedder de exemplo só 
para o pipeline não travar -- troque por embeddings reais assim que possível.

In [ ]:
from facetrust.embeddings import ArcFaceEmbedder, CallableEmbedder

try:
    embedder = ArcFaceEmbedder()
    print('Usando ArcFace (insightface).')
except Exception as e:
    print('ArcFace indisponível (', e, ') -- usando embedder de exemplo.')
    embedder = CallableEmbedder(lambda img: img.reshape(-1, img.shape[-1]).mean(axis=0))

## 2. Carregar o dataset

Estrutura esperada em `data/` (ver README do pacote):
```
data/clean/<subject_id>/<sample_id>.png
data/watermarked/<subject_id>/<sample_id>__alpha<alpha>.png
data/morph/<algorithm>/<contributor_a>+<contributor_b>.png
```
`<algorithm>` é o nome do método de morphing (ex. "stylegan2", "amsl", 
"facemorpher", "opencv" -- os quatro do FRLL-Morphs) -- vira 
`Sample.morph_algorithm`, o que habilita a comparação por algoritmo na Seção 6.

**Se algum sujeito só tiver 1 imagem `clean`:** a Seção 5 (verificação) pula 
essa condição com um aviso em vez de travar (não existe par genuine com só 
1 amostra), e a coluna RMMR na Seção 6 vem como `NaN` pelo mesmo motivo -- 
MMPMR/FMMPMR/ProdAvg continuam normalmente.

In [ ]:
from pathlib import Path
from demo_full_pipeline import load_dataset  # reaproveita o loader do script de exemplo

dataset = load_dataset(Path('data'))
print(f'{len(dataset.samples)} amostras carregadas.')
dataset.compute_embeddings(embedder)
print('Embeddings calculados.')

## 3. Pergunta da atividade

> Como o aumento da força do watermark ($\alpha$) muda, ao mesmo tempo, a 
> qualidade visual, a confiabilidade do watermark e a similaridade biométrica?

### Plot A: $\alpha$ vs. PSNR/SSIM  |  Plot B: $\alpha$ vs. embedding drift

*(As imagens em `data/watermarked/` deste dataset foram geradas OFFLINE com 
o SSLWatermarking de verdade -- ver `examples/ssl_watermarking_demo.py` -- 
não ao vivo: cada imagem passa por uma otimização iterativa, não um forward 
pass único, então não cabe nos 20-30 min da sessão.)*

In [ ]:
import matplotlib.pyplot as plt
from facetrust.pipeline import PipelineEvaluator
from facetrust.plotting import plot_alpha_sweep

evaluator = PipelineEvaluator(dataset)

iq = evaluator.image_quality_matrix()
drift = evaluator.embedding_drift_report('watermarked')

iq_by_alpha = iq[iq['condition'] == 'watermarked'].dropna(subset=['alpha']).groupby('alpha')[['psnr', 'ssim']].mean()
drift_by_alpha = drift.dropna(subset=['alpha']).groupby('alpha')['embedding_drift'].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_alpha_sweep(iq_by_alpha.index, iq_by_alpha['psnr'], ax=axes[0], ylabel='PSNR / SSIM', label='PSNR')
plot_alpha_sweep(iq_by_alpha.index, iq_by_alpha['ssim'], ax=axes[0], ylabel='PSNR / SSIM', label='SSIM')
plot_alpha_sweep(drift_by_alpha.index, drift_by_alpha.values, ax=axes[1], ylabel='Embedding drift')
axes[0].set_title('Plot A'); axes[1].set_title('Plot B')
fig.tight_layout(); plt.show()

iq_by_alpha.join(drift_by_alpha)

## 4. Robustez a transformações benignas

Repete a comparação após JPEG / resize / blur / crop / rotação -- a tabela 
"Robustness transformations" dos slides. ("Screen recapture" fica de fora: 
é discussão qualitativa, não sai do notebook -- precisa de captura física real.)

In [ ]:
robustness = evaluator.robustness_table(embedder, condition='watermarked')
robustness

## 5. Quatro cenários de comparação

Clean↔Clean, Clean↔Watermarked, Watermarked↔Clean, Watermarked↔Watermarked.

In [ ]:
scenarios = evaluator.four_scenario_report()
for name, pairs in scenarios.items():
    g = pairs.genuine.mean() if len(pairs.genuine) else float('nan')
    i = pairs.impostor.mean() if len(pairs.impostor) else float('nan')
    print(f'{name:28s} genuine={g:.4f}  impostor={i:.4f}')

## 5b. Verificação isolada (EER / TAR@FAR)

Só funciona para uma condição se ela tiver pelo menos 2 imagens por sujeito 
(senão não existe par genuine para medir EER/FRR) -- por isso o `try/except` abaixo: numa sessão ao vivo com poucas fotos por pessoa, é comum alguma 
condição não ter dados suficientes, e isso não deve travar o notebook.

In [ ]:
for condition in ['clean', 'watermarked']:
    if not dataset.by_condition(condition):
        continue
    try:
        report = evaluator.verification_report(condition)
        report.pop('scores')
        print(f'[{condition}]', report)
    except ValueError as exc:
        print(f'[{condition}] pulado -- {exc}')

## 6. Potencial de ataque de morphing (MMPMR / FMMPMR / ProdAvg-MMPMR / RMMR)

Threshold calibrado no FAR operacional do sistema (não em um valor arbitrário).

- **MMPMR**: pelo menos 1 prova de cada contribuidor engana o sistema.
- **FMMPMR**: TODAS as provas de TODOS os contribuidores enganam (mais estrita; FMMPMR <= MMPMR sempre).
- **ProdAvg-MMPMR**: variante contínua, menos enviesada quando há várias provas por contribuidor.
- **RMMR = MMPMR + FRR**: ancora a vulnerabilidade ao custo de usabilidade do mesmo threshold (pode ser `NaN` se faltar par genuine -- ver Seção 5b).

In [ ]:
from facetrust.plotting import plot_map_curve

morph_df = evaluator.morph_attack_report()
display(morph_df)

fig, ax = plt.subplots(figsize=(5, 4))
plot_map_curve(morph_df['target_far'], morph_df['mmpmr'], ax=ax, metric_name='Taxa de sucesso', label='MMPMR')
plot_map_curve(morph_df['target_far'], morph_df['fmmpmr'], ax=ax, metric_name='Taxa de sucesso', label='FMMPMR')
plot_map_curve(morph_df['target_far'], morph_df['prodavg_mmpmr'], ax=ax, metric_name='Taxa de sucesso', label='ProdAvg')
fig.tight_layout(); plt.show()

### 6b. Comparando algoritmos de morphing (se o dataset tiver mais de um)

Com o FRLL-Morphs, por exemplo: AMSL vs. FaceMorpher vs. OpenCV vs. StyleGAN2 no mesmo par de contribuidores.

In [ ]:
algorithms = {s.morph_algorithm for s in dataset.by_condition('morphed')}
if len(algorithms) > 1:
    morph_by_alg = evaluator.morph_attack_report(group_by_algorithm=True)
    display(morph_by_alg)

    fig, ax = plt.subplots(figsize=(6, 4.5))
    for algorithm, group in morph_by_alg.groupby('morph_algorithm'):
        plot_map_curve(group['target_far'], group['mmpmr'], ax=ax, metric_name='MMPMR', label=algorithm)
    fig.tight_layout(); plt.show()

    iq_morph = iq[iq['condition'] == 'morphed']
    display(iq_morph.groupby('morph_algorithm')[['psnr', 'ssim']].mean())
else:
    print('Só 1 algoritmo de morphing no dataset -- pule esta célula ou adicione mais para comparar.')

## 7. Discussão (perguntas dos slides)

1. Em que $\alpha$ o watermark fica visivelmente perceptível?
2. Em que $\alpha$ o embedding drift se torna relevante?
3. A detecção do watermark falha antes ou depois da similaridade biométrica?
4. Vocês usariam o mesmo ponto de operação para enrollment e para verificação?
5. Existe um único $\alpha$ "ótimo", ou isso é uma superfície de trade-off?
6. Olhando o MMPMR: em qual FAR operacional o ataque de morphing deixa de ser preocupante?
7. Por que MMPMR e FMMPMR podem divergir tanto, mesmo para a mesma morph e o mesmo threshold?
8. Se dois sistemas relatam o mesmo MMPMR, o que o RMMR pode revelar que o MMPMR sozinho esconde?
9. (Se usaram FRLL-Morphs) Algum algoritmo de morphing é sistematicamente mais perigoso que os outros? Isso se reflete também na qualidade de imagem (PSNR/SSIM), ou os dois eixos discordam?